# 31 - Source 20-action execution-horizon worker 0

This is shard 0 of 2 for the 130-episode action-horizon pilot: 13 LIBERO-PRO suites
x 10 tasks x the first episode initialization. The model still generates a 50-action chunk, but
the rollout executes its first **20 actions** before observing and replanning.

Run `ARM="refinement"` first. It uses the established always-on K=5, Euler-steps `(3,4)`,
last-refinement policy. Each worker collects 65 rollouts and prints its current per-suite SR every
25 rollouts beside the exact matched historical 10-action unrefined source baseline. If this arm
is promising, change `ARM` to `"baseline"` in both workers to collect the matched 20-action
uncertainty-only control. The two arms have distinct resumable IDs; changing `ARM` cannot overwrite
the refinement rows. Worker 0 may use `EPISODE_LIMIT=1` for a smoke test and then restore `None`.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (SOURCE_ACTION_HORIZON_EXPERIMENT,
    load_bootstrap_manifest, run_source_action_horizon_worker)

drive.mount("/content/drive")

ARM = "refinement"  # run first; later change to "baseline" only if this arm is promising
N_ACTION_STEPS = 20
EPISODES_PER_TASK = 1  # 13 suites x 10 tasks x 1 initialization = 130 episodes
SHARD_COUNT = 2
SHARD_INDEX = 0
EPISODE_LIMIT = None  # worker-0 smoke: set 1 once, then restore None
EXPERIMENT = SOURCE_ACTION_HORIZON_EXPERIMENT
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json")
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest["source_model"] == PI05_REPO_ID, manifest["source_model"]
SOURCE_MODEL_REVISION = manifest["source_model_revision"]
assert SOURCE_MODEL_REVISION, "v2 manifest is missing source_model_revision"

print({"experiment": EXPERIMENT, "arm": ARM,
       "n_action_steps": N_ACTION_STEPS, "generated_chunk_size": 50,
       "episodes_per_task": EPISODES_PER_TASK,
       "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
       "episode_limit": EPISODE_LIMIT,
       "manifest_hash": manifest["manifest_hash"],
       "source_model_revision": SOURCE_MODEL_REVISION})
run_source_action_horizon_worker(
    arm=ARM, n_action_steps=N_ACTION_STEPS,
    episodes_per_task=EPISODES_PER_TASK, episode_limit=EPISODE_LIMIT,
    shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest["manifest_hash"],
    source_model_revision=SOURCE_MODEL_REVISION, experiment=EXPERIMENT)